# Comparison and alignment features

This notebook demonstrates the lightweight comparison API using synthetic `ModesArray` data. It covers common time-axis handling, peak alignment, phase alignment, and one shared candidate time-shift optimization.

In [ ]:
import numpy as np

import waveformtools.comparison  # installs ModesArray convenience methods
from waveformtools.comparison import AlignmentSpec, mode_match
from waveformtools.modes_array import ModesArray


def make_modes(phase=0.0, time_axis=None, time_shift=0.0, orbital_phase=0.0):
    if time_axis is None:
        time_axis = np.linspace(-10.0, 10.0, 256)
    source_time = time_axis - time_shift
    envelope = np.exp(-0.05 * source_time**2)
    carrier = np.exp(1j * 0.7 * source_time)

    modes = ModesArray(ell_max=2, time_axis=time_axis, spin_weight=-2)
    modes.create_modes_array(ell_max=2, data_len=len(time_axis))
    modes.set_mode_data(
        ell=2,
        emm=2,
        data=envelope * carrier * np.exp(1j * phase) * np.exp(2j * orbital_phase),
    )
    modes.set_mode_data(
        ell=2,
        emm=-2,
        data=0.3 * envelope * np.conjugate(carrier) * np.exp(1j * phase) * np.exp(-2j * orbital_phase),
    )
    return modes


reference = make_modes()
candidate = make_modes(phase=0.4, orbital_phase=0.25)


## Global and orbital phase alignment

In [ ]:
for phase_alignment in ["none", "global_complex", "orbital_phase", "orbital_phase_and_global"]:
    result = mode_match(
        reference,
        candidate,
        time_alignment="none",
        phase_alignment=phase_alignment,
        alignment=AlignmentSpec(orbital_phase_samples=721),
    )
    print(
        phase_alignment,
        "match=", result.match,
        "mismatch=", result.mismatch,
        "orbital_phase=", result.best_parameters["orbital_phase"],
    )


## Unequal time axes: crop or resample

In [ ]:
reference = make_modes(time_axis=np.linspace(-10.0, 10.0, 256))
candidate = make_modes(time_axis=np.linspace(-7.0, 13.0, 256), time_shift=3.0)

for time_domain_policy in ["crop_to_overlap", "resample_to_reference"]:
    result = mode_match(
        reference,
        candidate,
        time_alignment="peak_total_news_power",
        time_domain_policy=time_domain_policy,
        phase_alignment="global_complex",
    )
    print(
        time_domain_policy,
        "match=", result.match,
        "n_samples=", result.diagnostics["time_axis"]["n_samples"],
        "overlap=", result.diagnostics["time_axis"]["overlap"],
    )


## Optimizing one shared candidate time shift

In [ ]:
time_axis = np.linspace(-12.0, 12.0, 512)
reference = make_modes(time_axis=time_axis)
candidate = make_modes(time_axis=time_axis, time_shift=0.35)

result = mode_match(
    reference,
    candidate,
    time_alignment="none",
    time_domain_policy="resample_to_reference",
    phase_alignment="global_complex",
    optimize_time_shift=True,
    time_shift_bounds=(-0.7, 0.0),
)

print("match=", result.match)
print("mismatch=", result.mismatch)
print("best candidate_time_shift=", result.best_parameters["candidate_time_shift"])
print("optimizer=", result.optimizer)
